In [4]:
import pandas as pd
import numpy as np


In [5]:
df = pd.read_json('/tiktok_booktok_2025_data.jsonl', lines=True)

In [11]:
print(df.columns)

Index(['id', 'like_count', 'region_code', 'video_description',
       'favorites_count', 'hashtag_names', 'effect_ids', 'is_stem_verified',
       'playlist_id', 'username', 'video_duration', 'video_label',
       'comment_count', 'create_time', 'music_id', 'view_count',
       'video_mention_list', 'voice_to_text', 'hashtag_info_list',
       'share_count', 'sticker_info_list', 'effect_info_list', 'video_tag',
       'is_trending_music'],
      dtype='object')


# 1. Identifying trending sounds (TOP 5% by frequency)

In [ ]:
# 1. Identifying trending sounds (TOP 5% by frequency)
top_audio = df['music_id'].value_counts().nlargest(int(len(df['music_id'].unique()) * 0.05)).index
print("Trending Sounds (Top 5% by frequency) with their counts:")
for audio in top_audio:
    count = df[df['music_id'] == audio].shape[0]
    print(f"{audio}: {count}")

In [8]:
df['is_trending_music'] = df['music_id'].isin(top_audio).astype(int)
print(df['is_trending_music'].value_counts())

is_trending_music
0    37007
1    14263
Name: count, dtype: int64


In [13]:
df['is_trending_music']

,is_trending_music
0,1
1,1
2,0
3,1
4,0
...,...
51265,0
51266,1
51267,0
51268,0


# 2. Using Effects and Stickers

In [14]:
# 2. Using Effects and Stickers
df['has_effects'] = df['effect_ids'].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0)
df['effects_count'] = df['effect_ids'].apply(lambda x: len(x) if isinstance(x, list) else 0)
df['has_stickers'] = df['sticker_info_list'].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0)


In [15]:
print(df['has_effects'].value_counts())
print(df['has_stickers'].value_counts())
print(df['effects_count'].describe())

has_effects
0    45198
1     6072
Name: count, dtype: int64
has_stickers
1    29334
0    21936
Name: count, dtype: int64
count    51270.000000
mean         0.122157
std          0.339691
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.000000
Name: effects_count, dtype: float64


# 3. Categorizing Video Lengths


In [16]:
print(df['video_duration'].describe())

count    51270.000000
mean        22.735713
std         58.565714
min          0.000000
25%          0.000000
50%         11.000000
75%         17.000000
max       2319.000000
Name: video_duration, dtype: float64


In [17]:
bins = [0, 15, 60, np.inf]
labels = ['Short (<15s)', 'Medium (15-60s)', 'Long (>60s)']
df['duration_category'] = pd.cut(df['video_duration'], bins=bins, labels=labels)


# 4. Hashtag Strategy Analysis

In [18]:
# 4. Hashtag Strategy Analysis
df['hashtag_count'] = df['hashtag_names'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Determining the presence of niche tags (e.g., genre tags)
niche_tags = {'darkacademia', 'romantasy', 'fantasy', 'darkromance', 'enemiestolovers', 'mafiaromance'}
df['has_niche_hashtag'] = df['hashtag_names'].apply(
    lambda x: 1 if isinstance(x, list) and any(tag.lower() in niche_tags for tag in x) else 0
)


In [19]:
# Preview of newly created variables
print(df[['is_trending_music', 'has_effects', 'has_stickers', 'duration_category', 'has_niche_hashtag']].head())

   is_trending_music  has_effects  has_stickers duration_category  \
0                  1            0             0               NaN   
1                  1            0             1               NaN   
2                  0            1             1      Short (<15s)   
3                  1            0             0   Medium (15-60s)   
4                  0            0             0      Short (<15s)   

   has_niche_hashtag  
0                  0  
1                  0  
2                  0  
3                  0  
4                  0  
